# Lookback analysis: how the AIC-selected memory behaves

Uses the per-window log of `LSTM_AIC` (falling back to the `lookback_period` column of `AR_AIC`, whose selection is identical) to describe the AIC-selected lookback $p^*_{i,\tau}$:

1. its distribution across stocks and windows, and how often it sits at the bounds;
2. how often it changes from one re-estimation window to the next;
3. how its variation splits into persistent differences between stocks, a common drift over time (the sample grows), and the remainder, plus mean $p^*$ in the pre-specified sub-periods;
4. a figure of the cross-sectional median and interquartile range of $p^*$ over time, saved to `Paper/figures/`;
5. the compute cost of AIC selection relative to training the fixed-lookback grid (needs the window logs of the grid runs).

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SOURCE_MODEL = "LSTM_AIC"          # falls back to AR_AIC if LSTM_AIC has not been run
FALLBACK_MODEL = "AR_AIC"
GRID = [10, 21, 42, 63]
MAX_AIC_LAG = 63
ESTIMATE_TYPE = "QMLE-Trade"
VALUE_COLUMN = "Volatility"
DUPLICATE_POLICY = "drop"

# Pre-specified sub-periods (test dates). "Other years" is every test date
# outside the three named periods.
PERIODS = {
    "Full sample": None,
    "Early sample (2004-2006)": [("2004-01-01", "2006-12-31")],
    "Financial crisis (2008-2009)": [("2008-01-01", "2009-12-31")],
    "COVID-19 (2020)": [("2020-01-01", "2020-12-31")],
    "Other years": "complement",
}

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Results").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing Results/ and Data/.")


def natural_key(name: str):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


def discover_models(results_dir: Path, models=None, include_robustness=False):
    """Every Results/<name>/<name>.csv. Robustness runs (names ending in
    _refit<N>m, _aic<N>y, _resel<N>m or _seed<N>) are excluded unless
    requested or listed explicitly."""
    found = {}
    for folder in results_dir.iterdir():
        main_file = folder / f"{folder.name}.csv"
        if folder.is_dir() and main_file.is_file():
            found[folder.name] = main_file
    if models is not None:
        missing = [m for m in models if m not in found]
        if missing:
            print(f"Not yet run (skipped): {missing}")
        found = {m: found[m] for m in models if m in found}
    elif not include_robustness:
        found = {k: v for k, v in found.items()
                 if not re.search(r"_(refit\d+m|aic\d+y|resel\d+m|seed\d+)", k)}
    return dict(sorted(found.items(), key=lambda item: natural_key(item[0])))


def parse_source_dates(values):
    """Same date parser as NN/NN.ipynb."""
    text = values.astype(str).str.strip()
    slash_mask = text.str.contains("/", regex=False)
    parsed = pd.Series(pd.NaT, index=text.index, dtype="datetime64[ns]")
    parsed.loc[~slash_mask] = pd.to_datetime(text.loc[~slash_mask], format="%Y-%m-%d", errors="raise")
    parsed.loc[slash_mask] = pd.to_datetime(text.loc[slash_mask], dayfirst=True, errors="raise")
    return parsed


def load_prediction_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "date" not in frame.columns:
        raise ValueError(f"Missing date column in {path}")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    if frame["date"].duplicated().any():
        raise ValueError(f"Duplicate dates in {path}")
    frame = frame.set_index("date").sort_index()
    return frame.apply(pd.to_numeric, errors="coerce")


def load_actuals(tickers, data_dir: Path) -> pd.DataFrame:
    """Realised volatility, constructed exactly as in the model notebooks:
    same estimate type, same date parser, same duplicate policy."""
    series = []
    for ticker in tickers:
        path = data_dir / f"{ticker}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"No realised-volatility file for {ticker}: {path}")
        frame = pd.read_csv(path)
        frame = frame.loc[frame["Type"].eq(ESTIMATE_TYPE), ["Date", VALUE_COLUMN]].copy()
        if frame.empty:
            raise ValueError(f"No {ESTIMATE_TYPE!r} observations in {path}")
        frame["Date"] = parse_source_dates(frame["Date"])
        duplicated = frame["Date"].duplicated(keep=False)
        if duplicated.any():
            dates = sorted(frame.loc[duplicated, "Date"].dt.date.unique())
            if DUPLICATE_POLICY == "drop":
                frame = frame.loc[~duplicated]
                action = "treated as missing"
            elif DUPLICATE_POLICY == "first":
                frame = frame.drop_duplicates("Date", keep="first")
                action = "first observation kept"
            else:
                raise ValueError("DUPLICATE_POLICY must be 'drop' or 'first'")
            print(f"Data note - {ticker}: repeated observations on {len(dates)} date(s) {action}: {dates}")
        item = pd.to_numeric(frame[VALUE_COLUMN], errors="coerce")
        item.index = frame["Date"]
        item.name = ticker
        series.append(item)
    return pd.concat(series, axis=1, sort=True)

def period_mask(dates, period):
    """Boolean mask over a DatetimeIndex for one entry of PERIODS."""
    dates = pd.DatetimeIndex(dates)
    if period is None:
        return np.ones(len(dates), dtype=bool)
    if isinstance(period, str) and period == "complement":
        named = np.zeros(len(dates), dtype=bool)
        for other in PERIODS.values():
            if isinstance(other, list):
                named |= period_mask(dates, other)
        return ~named
    mask = np.zeros(len(dates), dtype=bool)
    for start, end in period:
        mask |= (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))
    return mask


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "Results"
DATA_DIR = ROOT / "Data"
FIG_DIR = ROOT / "Paper" / "figures"
OUTPUT_DIR = RESULTS_DIR / "Analysis" / "output"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_lookbacks():
    """One row per stock and test window: ticker, test_start, lookback."""
    log_path = RESULTS_DIR / SOURCE_MODEL / f"{SOURCE_MODEL}_windows.csv"
    if log_path.is_file():
        log = pd.read_csv(log_path, parse_dates=["test_start"])
        return log[["ticker", "test_start", "lookback"]], SOURCE_MODEL
    for model in [SOURCE_MODEL, FALLBACK_MODEL]:
        folder = RESULTS_DIR / model / "individual"
        if folder.is_dir():
            frames = [pd.read_csv(p, parse_dates=["date"]) for p in sorted(folder.glob("*.csv"))]
            frame = pd.concat(frames, ignore_index=True).dropna(subset=["lookback_period"])
            frame["test_start"] = frame["date"].dt.to_period("M").dt.to_timestamp()
            out = (frame.groupby(["ticker", "test_start"], as_index=False)["lookback_period"].first()
                   .rename(columns={"lookback_period": "lookback"}))
            return out, model
    raise FileNotFoundError("Run LSTM_AIC (NN/NN.ipynb) first.")


lookbacks, source = load_lookbacks()
lookbacks["lookback"] = lookbacks["lookback"].astype(int)
lookbacks = lookbacks.sort_values(["ticker", "test_start"]).reset_index(drop=True)
print(f"Lookbacks from {source}: {len(lookbacks):,} stock-windows, "
      f"{lookbacks['ticker'].nunique()} stocks, "
      f"{lookbacks['test_start'].min().date()} to {lookbacks['test_start'].max().date()}")

In [ ]:
# 1. Distribution
p = lookbacks["lookback"]
summary = pd.Series({
    "mean": p.mean(), "median": p.median(), "p10": p.quantile(0.10), "p90": p.quantile(0.90),
    "min": p.min(), "max": p.max(),
    "share at p=1": (p == 1).mean(), f"share at p_max={MAX_AIC_LAG}": (p == MAX_AIC_LAG).mean(),
    "share <= 10": (p <= 10).mean(), "share > 42": (p > 42).mean(),
})
display(summary.to_frame("p*"))
per_stock = lookbacks.groupby("ticker")["lookback"].agg(["mean", "median", "min", "max", "std"])
display(per_stock.sort_values("median"))

# 2. Stability between consecutive annual re-selections. p* is re-selected
# each January and held for the monthly windows, so compare one value per
# stock and year rather than consecutive monthly windows.
annual_p = (lookbacks.assign(year=lookbacks["test_start"].dt.year)
            .groupby(["ticker", "year"], as_index=False)["lookback"].first())
annual_p["change"] = annual_p.groupby("ticker")["lookback"].diff()
changes = annual_p.dropna(subset=["change"])
stability = pd.Series({
    "share of re-selections where p* changes": (changes["change"] != 0).mean(),
    "mean |change| when it changes": changes.loc[changes["change"] != 0, "change"].abs().mean(),
    "median |change| when it changes": changes.loc[changes["change"] != 0, "change"].abs().median(),
})
display(stability.to_frame("value"))

In [ ]:
# 3. Where does the variation in p* come from?
# Two-way decomposition of the stock-by-window panel: persistent differences
# between stocks (stock means), a common drift over time (window means), and
# the remainder. Shares of the total sum of squares.
panel = lookbacks.pivot(index="test_start", columns="ticker", values="lookback").astype(float)
grand = panel.stack().mean()
total_ss = ((panel - grand) ** 2).sum().sum()
stock_ss = (((panel.mean(axis=0) - grand) ** 2) * panel.notna().sum(axis=0)).sum()
time_ss = (((panel.mean(axis=1) - grand) ** 2) * panel.notna().sum(axis=1)).sum()
last_date = panel.index.max()
regime = pd.Series({
    "share of variation: between stocks": stock_ss / total_ss,
    "share of variation: common over time": time_ss / total_ss,
    "share of variation: remainder": 1 - (stock_ss + time_ss) / total_ss,
    "cross-stock median p*, first window": panel.iloc[0].median(),
    "cross-stock median p*, last window": panel.iloc[-1].median(),
    f"min p* across stocks, {last_date.date()}": panel.loc[last_date].min(),
    f"max p* across stocks, {last_date.date()}": panel.loc[last_date].max(),
})
display(regime.to_frame("value"))

by_period = pd.Series({
    name: lookbacks.loc[period_mask(lookbacks["test_start"], period), "lookback"].mean()
    for name, period in PERIODS.items()
}, name="mean p*")
display(by_period.to_frame())

pd.concat([summary.rename("value"), stability.rename("value"), regime.rename("value"),
           by_period.rename("value")]).to_csv(OUTPUT_DIR / "lookback_summary.csv", header=True)

In [ ]:
# 4. Figure: cross-sectional distribution of p* over time
monthly = lookbacks.groupby("test_start")["lookback"].quantile([0.25, 0.5, 0.75]).unstack()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.fill_between(monthly.index, monthly[0.25], monthly[0.75], step="post", alpha=0.25,
                color="tab:blue", linewidth=0, label="Interquartile range of $p^*$")
ax.step(monthly.index, monthly[0.5], where="post", color="tab:blue", linewidth=1.2,
        label="Median $p^*$")
for start, end in [("2008-01-01", "2009-12-31"), ("2020-01-01", "2020-12-31")]:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="grey", alpha=0.12, linewidth=0)
ax.set_ylabel("AIC-selected lookback (days)")
ax.set_ylim(0, MAX_AIC_LAG + 2)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "lookback_over_time.pdf")
fig.savefig(FIG_DIR / "lookback_over_time.png", dpi=200)
plt.show()
print(f"Saved figure to {FIG_DIR / 'lookback_over_time.pdf'}")

In [ ]:
# 5. Compute cost: AIC selection vs training the fixed-lookback grid
logs = {}
for model in [f"LSTM_{L}" for L in GRID] + ["LSTM_AIC"]:
    path = RESULTS_DIR / model / f"{model}_windows.csv"
    if path.is_file():
        logs[model] = pd.read_csv(path)
if len(logs) < len(GRID) + 1:
    print(f"Window logs found for {sorted(logs)}; need all of the grid and LSTM_AIC for the cost comparison.")
else:
    keys = ["ticker", "test_start"]
    grid_cost = sum(logs[f"LSTM_{L}"].set_index(keys)["train_seconds"] for L in GRID)
    aic_log = logs["LSTM_AIC"].set_index(keys)
    aic_cost = aic_log["train_seconds"] + aic_log["aic_seconds"]
    cost = pd.Series({
        "AIC selection, mean seconds per stock-window": aic_log["aic_seconds"].mean(),
        "LSTM_AIC training, mean seconds per stock-window": aic_log["train_seconds"].mean(),
        "Grid training (LSTM_VAL), mean seconds per stock-window": grid_cost.mean(),
        "Total cost ratio, grid / LSTM_AIC": grid_cost.sum() / aic_cost.sum(),
    })
    display(cost.to_frame("value"))
    cost.to_csv(OUTPUT_DIR / "compute_cost.csv", header=["value"])
    print("Timings are wall-clock per worker and depend on machine load; treat them as approximate.")